<a href="https://colab.research.google.com/github/ABDOUNDIAYE1602/AppMobileSourds_Muets/blob/main/Copie_de_WLASL_Sign_Language.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## REPRISE RAPIDE
### Execute UNIQUEMENT cette cellule puis saute a l'ETAPE 5
N'execute pas cette cellule si c'est la premiere fois.

In [ ]:
# ============================================================
# REPRISE RAPIDE
# Temps estime : 3 a 4 minutes
# ============================================================

# -- Etape A : Monter le Drive
from google.colab import drive
drive.mount('/content/drive')
print('Drive monte')

# -- Etape B : Installer les dependances
import subprocess, sys
print('Installation des dependances en cours...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'mediapipe==1.0.1', 'opencv-python-headless', 'tqdm', 'scikit-learn', 'matplotlib', 'seaborn', 'tensorflow==2.20.0'], check=True)
print('Dependances installees')

# -- Etape C : Imports
import os, json
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split

# -- Etape D : Chemins (adaptes a ta structure Drive)
DRIVE_ROOT   = '/content/drive/MyDrive'
PROJECT_DIR  = os.path.join(DRIVE_ROOT, 'ASL_PROJECT')
WLASL_DIR    = os.path.join(PROJECT_DIR, 'WLASL')
VIDEOS_DIR   = os.path.join(WLASL_DIR,   'videos')
JSON_PATH    = os.path.join(WLASL_DIR,   'WLASL_v0.3.json')
SPLIT_PATH = os.path.join(WLASL_DIR, 'nslt_100.json')

# Dossiers de travail crees par ce notebook
WORK_DIR     = os.path.join(PROJECT_DIR, 'WORK')
KEYPOINTS_DIR = os.path.join(WORK_DIR, 'keypoints')
MODELS_DIR   = os.path.join(WORK_DIR, 'models')
LOGS_DIR     = os.path.join(WORK_DIR, 'logs')

for d in [WORK_DIR, KEYPOINTS_DIR, MODELS_DIR, LOGS_DIR]:
    os.makedirs(d, exist_ok=True)

# -- Etape E : Charger la config sauvegardee
config_path = os.path.join(WORK_DIR, 'model_config.json')
if not os.path.exists(config_path):
    print('ERREUR : Aucune config trouvee. Lance les etapes 0 a 4 dabord.')
    raise SystemExit()

with open(config_path) as f:
    config = json.load(f)

LABELS        = config['labels']
NUM_CLASSES   = config['num_classes']
SEQUENCE_LEN  = config['sequence_len']
KEYPOINT_SIZE = config['keypoint_size']
NUM_SIGNS     = config['num_signs']
print(f'Config chargee : {NUM_CLASSES} signes, seq={SEQUENCE_LEN}, kp={KEYPOINT_SIZE}')

# -- Etape F : Charger les donnees extraites depuis Drive
x_path = os.path.join(WORK_DIR, 'X_full.npy')
y_path = os.path.join(WORK_DIR, 'y_full.npy')

if not os.path.exists(x_path):
    print('ERREUR : X_full.npy introuvable. Lance l\'Etape 3 dabord.')
    raise SystemExit()

X = np.load(x_path)
y = np.load(y_path)
y_cat = to_categorical(y, num_classes=NUM_CLASSES)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_cat, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42)

# Augmentation
np.random.seed(42)
X_aug = (X_train + np.random.normal(0, 0.005, X_train.shape)).astype(np.float32)
X_train_final = np.concatenate([X_train, X_aug], axis=0)
y_train_final = np.concatenate([y_train, y_train], axis=0)
idx = np.random.permutation(len(X_train_final))
X_train_final = X_train_final[idx]
y_train_final = y_train_final[idx]

print(f'Donnees chargees : {X.shape[0]} sequences au total')
print(f'Train : {X_train_final.shape[0]} | Val : {X_val.shape[0]} | Test : {X_test.shape[0]}')

# -- Etape G : Charger le meilleur modele sauvegarde
best_model_path = os.path.join(MODELS_DIR, 'best_model.h5')
if os.path.exists(best_model_path):
    model = load_model(best_model_path)
    model.compile(
        optimizer=Adam(learning_rate=5e-4),
        loss='categorical_crossentropy',
        metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=5, name='top5_acc')]
    )
    val_loss, val_acc, val_top5 = model.evaluate(X_val, y_val, verbose=0)
    print(f'Modele charge : {best_model_path}')
    print(f'Precision validation actuelle : {val_acc*100:.1f}%')
    print(f'Top-5 validation              : {val_top5*100:.1f}%')
else:
    print('Aucun modele sauvegarde trouve. Lance l\'Etape 5 pour entrainer.')

print('\nGPU disponible :', tf.config.list_physical_devices('GPU'))
print('\nReprise prete. Tu peux relancer l\'Etape 5 pour continuer l\'entrainement.')

## ETAPE 0 — Monter Google Drive et configurer les chemins

In [19]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Chemins de ton dataset (ne pas modifier sauf si ta structure est differente)
DRIVE_ROOT    = '/content/drive/MyDrive'
PROJECT_DIR   = os.path.join(DRIVE_ROOT,  'ASL_PROJECT')
WLASL_DIR     = os.path.join(PROJECT_DIR, 'WLASL')
VIDEOS_DIR    = os.path.join(WLASL_DIR,   'videos')
JSON_PATH     = os.path.join(WLASL_DIR,   'WLASL_v0.3.json')
SPLIT_PATH = os.path.join(WLASL_DIR, 'nslt_100.json')
# Dossiers de travail crees automatiquement
WORK_DIR      = os.path.join(PROJECT_DIR, 'WORK')
KEYPOINTS_DIR = os.path.join(WORK_DIR, 'keypoints')
MODELS_DIR    = os.path.join(WORK_DIR, 'models')
LOGS_DIR      = os.path.join(WORK_DIR, 'logs')

for d in [WORK_DIR, KEYPOINTS_DIR, MODELS_DIR, LOGS_DIR]:
    os.makedirs(d, exist_ok=True)

# Verification que les fichiers existent
print('Verification de la structure Drive :')
checks = {
    'Dossier WLASL'       : WLASL_DIR,
    'Dossier videos'      : VIDEOS_DIR,
    'WLASL_v0.3.json'     : JSON_PATH,
    'nslt_2000.json'      : SPLIT_PATH,
}
all_ok = True
for label, path in checks.items():
    exists = os.path.exists(path)
    status = 'OK' if exists else 'INTROUVABLE'
    print(f'  [{status}] {label} : {path}')
    if not exists:
        all_ok = False

if all_ok:
    print('\nTous les fichiers sont accessibles. Tu peux continuer.')
else:
    print('\nATTENTION : Certains fichiers sont manquants. Verifie ta structure Drive.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Verification de la structure Drive :
  [OK] Dossier WLASL : /content/drive/MyDrive/ASL_PROJECT/WLASL
  [OK] Dossier videos : /content/drive/MyDrive/ASL_PROJECT/WLASL/videos
  [OK] WLASL_v0.3.json : /content/drive/MyDrive/ASL_PROJECT/WLASL/WLASL_v0.3.json
  [OK] nslt_2000.json : /content/drive/MyDrive/ASL_PROJECT/WLASL/nslt_100.json

Tous les fichiers sont accessibles. Tu peux continuer.


In [20]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Chemins de ton dataset (ne pas modifier sauf si ta structure est differente)
DRIVE_ROOT    = '/content/drive/MyDrive'
PROJECT_DIR   = os.path.join(DRIVE_ROOT,  'ASL_PROJECT')
WLASL_DIR     = os.path.join(PROJECT_DIR, 'WLASL')
VIDEOS_DIR    = os.path.join(WLASL_DIR,   'videos')
JSON_PATH     = os.path.join(WLASL_DIR,   'WLASL_v0.3.json')
SPLIT_PATH = os.path.join(WLASL_DIR, 'nslt_100.json')
# Dossiers de travail crees automatiquement
WORK_DIR      = os.path.join(PROJECT_DIR, 'WORK')
KEYPOINTS_DIR = os.path.join(WORK_DIR, 'keypoints')
MODELS_DIR    = os.path.join(WORK_DIR, 'models')
LOGS_DIR      = os.path.join(WORK_DIR, 'logs')

for d in [WORK_DIR, KEYPOINTS_DIR, MODELS_DIR, LOGS_DIR]:
    os.makedirs(d, exist_ok=True)

# Verification que les fichiers existent
print('Verification de la structure Drive :')
checks = {
    'Dossier WLASL'       : WLASL_DIR,
    'Dossier videos'      : VIDEOS_DIR,
    'WLASL_v0.3.json'     : JSON_PATH,
    'nslt_2000.json'      : SPLIT_PATH,
}
all_ok = True
for label, path in checks.items():
    exists = os.path.exists(path)
    status = 'OK' if exists else 'INTROUVABLE'
    print(f'  [{status}] {label} : {path}')
    if not exists:
        all_ok = False

if all_ok:
    print('\nTous les fichiers sont accessibles. Tu peux continuer.')
else:
    print('\nATTENTION : Certains fichiers sont manquants. Verifie ta structure Drive.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Verification de la structure Drive :
  [OK] Dossier WLASL : /content/drive/MyDrive/ASL_PROJECT/WLASL
  [OK] Dossier videos : /content/drive/MyDrive/ASL_PROJECT/WLASL/videos
  [OK] WLASL_v0.3.json : /content/drive/MyDrive/ASL_PROJECT/WLASL/WLASL_v0.3.json
  [OK] nslt_2000.json : /content/drive/MyDrive/ASL_PROJECT/WLASL/nslt_100.json

Tous les fichiers sont accessibles. Tu peux continuer.


## ETAPE 1 — Installation des dependances

In [21]:
!pip install mediapipe==1.0.1 tensorflow

import tensorflow as tf
import mediapipe as mp
import cv2, os, json, numpy as np
from tqdm import tqdm


print('TensorFlow :', tf.__version__)
print('MediaPipe  :', mp.__version__)

TensorFlow : 2.20.0
MediaPipe  : 1.0.1


## ETAPE 2 — Lecture du dataset et selection des signes

In [22]:
import json
import os

# ============================================================
# CONFIGURATION — Modifie uniquement cette section
# ============================================================
NUM_SIGNS    = 2000   # Nombre de signes a utiliser
SEQUENCE_LEN = 30    # Frames par sequence
MIN_VIDEOS   = 3     # Minimum de videos par signe
# ============================================================

# Charger le fichier JSON principal
with open(JSON_PATH, 'r') as f:
    wlasl_data = json.load(f)

# Charger les splits officiels (nslt_2000.json)
with open(SPLIT_PATH, 'r') as f:
    split_data = json.load(f)

print(f'Total signes dans WLASL_v0.3.json : {len(wlasl_data)}')
print(f'Total entrees dans nslt_2000.json  : {len(split_data)}')

# Construire un index video_id -> split (train/val/test)
# nslt_2000.json format : { "video_id": {"action": [...], "subset": "train"} }
video_split_map = {}
for vid_id, info in split_data.items():
    video_split_map[vid_id] = info.get('subset', 'train')

print(f'\nDistribution des splits dans nslt_2000.json :')
from collections import Counter
split_counts = Counter(video_split_map.values())
for split, count in split_counts.items():
    print(f'  {split} : {count} videos')

Total signes dans WLASL_v0.3.json : 2000
Total entrees dans nslt_2000.json  : 2038

Distribution des splits dans nslt_2000.json :
  train : 1442 videos
  val : 338 videos
  test : 258 videos


In [23]:
# Selectionner les signes qui ont assez de videos disponibles localement
selected_signs = []
missing_videos = 0
found_videos   = 0

for entry in wlasl_data:
    gloss     = entry['gloss']
    instances = entry.get('instances', [])

    # Verifier combien de videos sont reellement presentes sur Drive
    available = []
    for inst in instances:
        vid_id    = str(inst.get('video_id', ''))
        # Les videos peuvent etre sous videos/gloss/video_id.mp4
        # ou directement videos/video_id.mp4 selon la structure
        path_flat  = os.path.join(VIDEOS_DIR, f'{vid_id}.mp4')
        path_sub   = os.path.join(VIDEOS_DIR, gloss, f'{vid_id}.mp4')
        if os.path.exists(path_flat):
            available.append(('flat', vid_id, path_flat))
        elif os.path.exists(path_sub):
            available.append(('sub', vid_id, path_sub))

    if len(available) >= MIN_VIDEOS:
        selected_signs.append({'gloss': gloss, 'videos': available})
        found_videos += len(available)
    else:
        missing_videos += len(instances) - len(available)

    if len(selected_signs) >= NUM_SIGNS:
        break

LABELS     = [s['gloss'] for s in selected_signs]
NUM_CLASSES = len(LABELS)

# Sauvegarder la config
config = {
    'num_signs'    : NUM_SIGNS,
    'num_classes'  : NUM_CLASSES,
    'sequence_len' : SEQUENCE_LEN,
    'keypoint_size': 258,
    'labels'       : LABELS
}
with open(os.path.join(WORK_DIR, 'model_config.json'), 'w') as f:
    json.dump(config, f, indent=2)

KEYPOINT_SIZE = 258

print(f'Signes selectionnes    : {NUM_CLASSES}')
print(f'Videos disponibles     : {found_videos}')
print(f'Videos manquantes      : {missing_videos}')
print(f'\nPremiers signes        : {LABELS[:10]}')
print(f'\nConfig sauvegardee dans : {WORK_DIR}/model_config.json')

Signes selectionnes    : 1986
Videos disponibles     : 11952
Videos manquantes      : 75

Premiers signes        : ['book', 'drink', 'computer', 'before', 'chair', 'go', 'clothes', 'who', 'candy', 'cousin']

Config sauvegardee dans : /content/drive/MyDrive/ASL_PROJECT/WORK/model_config.json


## ETAPE 3 — Extraction des keypoints MediaPipe
Cette etape est longue (plusieurs heures pour 100 signes).


In [24]:
import cv2
import numpy as np
import mediapipe as mp
import urllib.request, os

# Telecharger les modeles necessaires
POSE_MODEL_PATH = '/tmp/pose_landmarker.task'
HAND_MODEL_PATH = '/tmp/hand_landmarker.task'

if not os.path.exists(POSE_MODEL_PATH):
    urllib.request.urlretrieve(
        'https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/latest/pose_landmarker_lite.task',
        POSE_MODEL_PATH
    )
    print('Modele pose telecharge')

if not os.path.exists(HAND_MODEL_PATH):
    urllib.request.urlretrieve(
        'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task',
        HAND_MODEL_PATH
    )
    print('Modele mains telecharge')

from mediapipe.tasks.python import vision
from mediapipe.tasks.python.vision import PoseLandmarker, PoseLandmarkerOptions
from mediapipe.tasks.python.vision import HandLandmarker, HandLandmarkerOptions
from mediapipe.tasks import python as mp_tasks

KEYPOINT_SIZE = 258

def extract_keypoints_from_results(pose_result, hand_result):
    # Pose : 33 points x 4 = 132
    if pose_result.pose_landmarks:
        pose = np.array([[lm.x, lm.y, lm.z, lm.visibility]
                         for lm in pose_result.pose_landmarks[0]]).flatten()
    else:
        pose = np.zeros(132)

    # Mains : on recupere gauche et droite
    lh = np.zeros(63)
    rh = np.zeros(63)
    if hand_result.hand_landmarks:
        for i, hand_landmarks in enumerate(hand_result.hand_landmarks):
            handedness = hand_result.handedness[i][0].category_name
            kp = np.array([[lm.x, lm.y, lm.z]
                            for lm in hand_landmarks]).flatten()
            if handedness == 'Left':
                lh = kp
            else:
                rh = kp

    return np.concatenate([lh, rh, pose])


def video_to_keypoints(video_path, target_frames=30):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None

    # Options pose
    pose_opts = PoseLandmarkerOptions(
        base_options=mp_tasks.BaseOptions(model_asset_path=POSE_MODEL_PATH),
        running_mode=vision.RunningMode.IMAGE
    )
    hand_opts = HandLandmarkerOptions(
        base_options=mp_tasks.BaseOptions(model_asset_path=HAND_MODEL_PATH),
        running_mode=vision.RunningMode.IMAGE,
        num_hands=2
    )

    frames_kp = []
    with PoseLandmarker.create_from_options(pose_opts) as pose_det, \
         HandLandmarker.create_from_options(hand_opts) as hand_det:
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            rgb    = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            pose_result = pose_det.detect(mp_img)
            hand_result = hand_det.detect(mp_img)
            frames_kp.append(extract_keypoints_from_results(pose_result, hand_result))
    cap.release()

    if len(frames_kp) == 0:
        return None

    frames_kp = np.array(frames_kp)
    N = len(frames_kp)

    if N == target_frames:
        return frames_kp
    elif N > target_frames:
        idx = np.linspace(0, N - 1, target_frames, dtype=int)
        return frames_kp[idx]
    else:
        x_old = np.linspace(0, 1, N)
        x_new = np.linspace(0, 1, target_frames)
        return np.array([
            np.interp(x_new, x_old, frames_kp[:, i])
            for i in range(KEYPOINT_SIZE)
        ]).T

print('Fonctions extraction prates.')
print(f'Keypoints par frame : {KEYPOINT_SIZE}')
print(f'Frames par sequence : {SEQUENCE_LEN}')

Modele pose telecharge
Modele mains telecharge
Fonctions extraction prates.
Keypoints par frame : 258
Frames par sequence : 30


In [ ]:
from tqdm import tqdm

label_to_idx     = {label: i for i, label in enumerate(LABELS)}
all_sequences    = []
all_labels       = []
stats = {'success': 0, 'failed': 0, 'cached': 0}

for sign_entry in tqdm(selected_signs, desc='Extraction keypoints'):
    gloss     = sign_entry['gloss']
    label_idx = label_to_idx[gloss]

    # Fichiers cache sur Drive
    kp_cache  = os.path.join(KEYPOINTS_DIR, f'{gloss}.npy')
    lbl_cache = os.path.join(KEYPOINTS_DIR, f'{gloss}_labels.npy')

    # Si deja extraits : charger depuis Drive directement
    if os.path.exists(kp_cache) and os.path.exists(lbl_cache):
        seqs = np.load(kp_cache)
        lbls = np.load(lbl_cache)
        all_sequences.extend(seqs)
        all_labels.extend(lbls)
        stats['cached'] += len(seqs)
        continue

    # Sinon : extraire depuis les videos
    sign_seqs = []
    sign_lbls = []

    for _, vid_id, video_path in sign_entry['videos']:
        kp_seq = video_to_keypoints(video_path, target_frames=SEQUENCE_LEN)
        if kp_seq is not None and kp_seq.shape == (SEQUENCE_LEN, KEYPOINT_SIZE):
            sign_seqs.append(kp_seq)
            sign_lbls.append(label_idx)
            stats['success'] += 1
        else:
            stats['failed'] += 1

    # Sauvegarder sur Drive pour ne pas refaire
    if sign_seqs:
        np.save(kp_cache,  np.array(sign_seqs))
        np.save(lbl_cache, np.array(sign_lbls))
        all_sequences.extend(sign_seqs)
        all_labels.extend(sign_lbls)

# Assembler le dataset complet
X = np.array(all_sequences, dtype=np.float32)
y = np.array(all_labels,    dtype=np.int32)

# Sauvegarder le dataset complet sur Drive
np.save(os.path.join(WORK_DIR, 'X_full.npy'), X)
np.save(os.path.join(WORK_DIR, 'y_full.npy'), y)

print(f'\nExtraction terminee :')
print(f'  Extraits avec succes : {stats["success"]}')
print(f'  Charges depuis cache : {stats["cached"]}')
print(f'  Echecs               : {stats["failed"]}')
print(f'\nDataset sauvegarde :')
print(f'  X shape : {X.shape}  ({X.shape[0]} sequences x {X.shape[1]} frames x {X.shape[2]} keypoints)')
print(f'  y shape : {y.shape}')

Extraction keypoints:  15%|█▌        | 303/1986 [4:58:51<23:12:37, 49.65s/it]

## ETAPE 4 — Preparation des donnees : split et augmentation

In [ ]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt
import seaborn as sns

# One-hot encoding
y_cat = to_categorical(y, num_classes=NUM_CLASSES)

# Split 70% train / 15% validation / 15% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_cat, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42)

print(f'Split du dataset :')
print(f'  Train      : {X_train.shape[0]} sequences  ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'  Validation : {X_val.shape[0]} sequences  ({X_val.shape[0]/len(X)*100:.0f}%)')
print(f'  Test       : {X_test.shape[0]} sequences  ({X_test.shape[0]/len(X)*100:.0f}%)')
print(f'  Classes    : {NUM_CLASSES}')

# Distribution des classes
fig, ax = plt.subplots(figsize=(18, 4))
class_counts = np.bincount(y)
ax.bar(range(len(LABELS)), class_counts, color='steelblue', alpha=0.8)
ax.set_xticks(range(len(LABELS)))
ax.set_xticklabels(LABELS, rotation=90, fontsize=6)
ax.set_title('Nombre de videos par signe', fontsize=13)
ax.set_ylabel('Nombre de videos')
plt.tight_layout()
plt.savefig(os.path.join(LOGS_DIR, 'class_distribution.png'), dpi=150)
plt.show()
print('Graphique sauvegarde dans WORK/logs/')

In [ ]:
# Augmentation : ajouter du bruit gaussien pour regulariser le modele
np.random.seed(42)

def augment_sequence(seq, noise_std=0.005):
    return seq + np.random.normal(0, noise_std, seq.shape).astype(np.float32)

X_aug = np.array([augment_sequence(seq) for seq in X_train])
X_train_final = np.concatenate([X_train, X_aug], axis=0)
y_train_final = np.concatenate([y_train, y_train], axis=0)

idx = np.random.permutation(len(X_train_final))
X_train_final = X_train_final[idx]
y_train_final = y_train_final[idx]

print(f'Apres augmentation :')
print(f'  Train original  : {X_train.shape[0]}')
print(f'  Train augmente  : {X_train_final.shape[0]} (x2)')

Apres augmentation :
  Train original  : 709
  Train augmente  : 1418 (x2)


## ETAPE 5 — Construction et entrainement du modele BiLSTM + Attention

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Bidirectional, LSTM, Dense, Dropout,
    BatchNormalization, GlobalAveragePooling1D,
    MultiHeadAttention, LayerNormalization, Add
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping,
    ReduceLROnPlateau, TensorBoard
)
import datetime

print('GPU disponible :', tf.config.list_physical_devices('GPU'))


def build_model(num_classes, seq_len=30, input_size=258):
    """
    Architecture BiLSTM + Self-Attention
    Entree : (batch, seq_len, input_size)
    Sortie : (batch, num_classes)
    """
    inputs = Input(shape=(seq_len, input_size), name='keypoints_input')

    # Couche 1 : BiLSTM
    # Using recurrent_activation='sigmoid' and unroll=True to force non-CuDNN LSTM, which is TFLite compatible
    x = Bidirectional(LSTM(256, return_sequences=True, recurrent_activation='sigmoid', unroll=True))(inputs)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)

    # Couche 2 : BiLSTM
    # Using recurrent_activation='sigmoid' and unroll=True to force non-CuDNN LSTM, which is TFLite compatible
    x = Bidirectional(LSTM(128, return_sequences=True, recurrent_activation='sigmoid', unroll=True))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)

    # Couche 3 : Self-Attention
    attn = MultiHeadAttention(num_heads=4, key_dim=32)(x, x)
    x    = LayerNormalization()(Add()([x, attn]))

    # Agregation temporelle
    x = GlobalAveragePooling1D()(x)

    # Couches denses
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.4)(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.3)(x)

    outputs = Dense(num_classes, activation='softmax', name='predictions')(x)

    model = Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss='categorical_crossentropy',
        metrics=[
            'accuracy',
            tf.keras.metrics.TopKCategoricalAccuracy(k=5, name='top5_acc')
        ]
    )
    return model


model = build_model(NUM_CLASSES, SEQUENCE_LEN, KEYPOINT_SIZE)
model.summary()
print(f'\nModele construit : {model.count_params():,} parametres')

In [ ]:
best_model_path = os.path.join(MODELS_DIR, 'best_model.h5')
log_dir = os.path.join(LOGS_DIR, datetime.datetime.now().strftime('%Y%m%d-%H%M%S'))

callbacks = [
    # Sauvegarder automatiquement le meilleur modele sur Drive
    ModelCheckpoint(
        best_model_path,
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    # Arreter si pas d'amelioration pendant 15 epochs
    EarlyStopping(
        monitor='val_accuracy',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
    # Reduire le learning rate si stagnation
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=7,
        min_lr=1e-6,
        verbose=1
    ),
    TensorBoard(log_dir=log_dir)
]

print('Debut de l\'entrainement...')
print(f'Modele sauvegarde automatiquement dans : {best_model_path}')
print()

history = model.fit(
    X_train_final, y_train_final,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

print('\nEntrainement termine.')

## ETAPE 6 — Evaluation des resultats

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

# Courbes d'entrainement
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'],     label='Train', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Validation', linewidth=2)
axes[0].set_title('Accuracy par epoch')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['loss'],     label='Train', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Validation', linewidth=2)
axes[1].set_title('Loss par epoch')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(LOGS_DIR, 'training_curves.png'), dpi=150)
plt.show()

# Score final sur le set de test
test_loss, test_acc, test_top5 = model.evaluate(X_test, y_test, verbose=0)
print(f'\nResultats finaux sur le set de TEST :')
print(f'  Accuracy  : {test_acc*100:.2f}%')
print(f'  Top-5 Acc : {test_top5*100:.2f}%')
print(f'  Loss      : {test_loss:.4f}')

In [ ]:
# Matrice de confusion (affichage des 25 premiers signes max)
y_pred     = model.predict(X_test, verbose=0)
y_pred_cls = np.argmax(y_pred, axis=1)
y_true_cls = np.argmax(y_test, axis=1)

# Get unique labels actually present in y_true_cls
unique_true_labels = np.unique(y_true_cls)
# Filter LABELS to match only the present unique true labels for target_names
filtered_target_names = [LABELS[i] for i in unique_true_labels]

max_display = min(NUM_CLASSES, 25)
mask = y_true_cls < max_display
cm   = confusion_matrix(y_true_cls[mask], y_pred_cls[mask])

fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=LABELS[:max_display],
    yticklabels=LABELS[:max_display],
    ax=ax
)
ax.set_xlabel('Predit',  fontsize=11)
ax.set_ylabel('Reel',    fontsize=11)
ax.set_title(f'Matrice de Confusion (premiers {max_display} signes)', fontsize=13)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0,  fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(LOGS_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()

print('Rapport de classification :')
print(classification_report(
    y_true_cls, y_pred_cls,
    labels=unique_true_labels, # Specify the actual labels present
    target_names=filtered_target_names,
    zero_division=0
))

## ETAPE 7 — Export du modele et du script PC

In [ ]:
# Format Keras natif
final_keras = os.path.join(MODELS_DIR, 'sign_language_final.keras')
model.save(final_keras)
print(f'Modele Keras sauvegarde  : {final_keras}')

# Format TFLite
# Convertir le modele Keras en une ConcreteFunction pour une meilleure compatibilite TFLite
concrete_func = tf.function(lambda inputs: model(inputs)).get_concrete_function(
    tf.TensorSpec(model.inputs[0].shape, model.inputs[0].dtype)
)
converter = tf.lite.TFLiteConverter.from_concrete_functions([concrete_func])
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()
final_tflite = os.path.join(MODELS_DIR, 'sign_language.tflite')
with open(final_tflite, 'wb') as f:
    f.write(tflite_model)
print(f'Modele TFLite sauvegarde : {final_tflite}')

# Mettre a jour la config
config.update({
    'test_accuracy' : float(test_acc),
    'top5_accuracy' : float(test_top5),
    'test_loss'     : float(test_loss)
})
config_path = os.path.join(WORK_DIR, 'model_config.json')
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)
print(f'Config mise a jour        : {config_path}')

print(f'\nFichiers a telecharger sur ton PC :')
print(f'  1. {final_keras}')
print(f'  2. {config_path}')

In [ ]:
# Generer et sauvegarder le script de detection PC

detect_script = '''
# detect_signs.py
# Detection langue des signes en temps reel via webcam
#
# Installation : pip install tensorflow mediapipe opencv-python numpy
# Lancement    : python detect_signs.py
#
# Fichiers necessaires dans le meme dossier :
#   - sign_language_final.h5
#   - model_config.json

import cv2
import mediapipe as mp
import numpy as np
import json
from collections import deque
from tensorflow.keras.models import load_model

# -- Configuration
MODEL_PATH            = \'sign_language_final.h5\'
CONFIG_PATH           = \'model_config.json\'
CONFIDENCE_THRESHOLD  = 0.65

# -- Chargement
print("Chargement du modele...")
model = load_model(MODEL_PATH)
with open(CONFIG_PATH) as f:
    config = json.load(f)

LABELS        = config[\'labels\']
SEQUENCE_LEN  = config[\'sequence_len\']
KEYPOINT_SIZE = config[\'keypoint_size\']
print(f"Modele charge : {len(LABELS)} signes | Precision test : {config.get(\'test_accuracy\', 0)*100:.1f}%")

# -- MediaPipe
mp_holistic    = mp.solutions.holistic
mp_drawing     = mp.solutions.drawing_utils
mp_draw_styles = mp.solutions.drawing_styles

def extract_keypoints(results):
    lh   = np.array([[lm.x, lm.y, lm.z] for lm in results.left_hand_landmarks.landmark]).flatten() \\
           if results.left_hand_landmarks else np.zeros(63)
    rh   = np.array([[lm.x, lm.y, lm.z] for lm in results.right_hand_landmarks.landmark]).flatten() \\
           if results.right_hand_landmarks else np.zeros(63)
    pose = np.array([[lm.x, lm.y, lm.z, lm.visibility]
                     for lm in results.pose_landmarks.landmark]).flatten() \\
           if results.pose_landmarks else np.zeros(132)
    return np.concatenate([lh, rh, pose])

# -- Webcam
cap           = cv2.VideoCapture(0)
sequence      = deque(maxlen=SEQUENCE_LEN)
current_sign  = ""
current_conf  = 0.0
history_signs = deque(maxlen=5)

print("Webcam active. Appuie sur Q pour quitter.")

with mp_holistic.Holistic(
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as holistic:

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame  = cv2.flip(frame, 1)
        h, w   = frame.shape[:2]

        # Traitement MediaPipe
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        rgb.flags.writeable = False
        results = holistic.process(rgb)
        rgb.flags.writeable = True

        # Dessiner les landmarks mains et corps
        mp_drawing.draw_landmarks(
            frame, results.left_hand_landmarks,
            mp.solutions.hands.HAND_CONNECTIONS,
            mp_draw_styles.get_default_hand_landmarks_style()
        )
        mp_drawing.draw_landmarks(
            frame, results.right_hand_landmarks,
            mp.solutions.hands.HAND_CONNECTIONS,
            mp_draw_styles.get_default_hand_landmarks_style()
        )
        mp_drawing.draw_landmarks(
            frame, results.pose_landmarks,
            mp.solutions.pose.POSE_CONNECTIONS
        )

        # Accumuler les keypoints
        kp = extract_keypoints(results)
        sequence.append(kp)

        # Predire quand la sequence est complete
        if len(sequence) == SEQUENCE_LEN:
            inp     = np.expand_dims(list(sequence), axis=0).astype(np.float32)
            preds   = model.predict(inp, verbose=0)[0]
            top_idx = np.argmax(preds)
            top_conf= preds[top_idx]

            if top_conf >= CONFIDENCE_THRESHOLD:
                current_sign = LABELS[top_idx]
                current_conf = top_conf
                if not history_signs or history_signs[-1] != current_sign:
                    history_signs.append(current_sign)

        # Interface
        overlay = frame.copy()
        cv2.rectangle(overlay, (0, 0), (w, 80), (0, 0, 0), -1)
        cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)

        if current_sign:
            color = (0, 255, 0) if current_conf > 0.85 else (0, 200, 255)
            cv2.putText(frame, f\'Signe : {current_sign}\',
                        (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.1, color, 2)
            cv2.putText(frame, f\'{current_conf*100:.0f}%\',
                        (w - 90, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, 2)

        # Barre de progression de la sequence
        progress = int(len(sequence) / SEQUENCE_LEN * w)
        cv2.rectangle(frame, (0, h - 8), (progress, h), (0, 180, 255), -1)

        # Historique des signes detectes
        hist = \'  ->  \'.join(list(history_signs))
        cv2.putText(frame, hist, (10, h - 18),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (200, 200, 200), 1)

        cv2.imshow(\'Sign Language Detection\', frame)

        if cv2.waitKey(1) & 0xFF == ord(\'q\'):
            break

cap.release()
cv2.destroyAllWindows()
print("Session terminee.")
'''

script_path = os.path.join(WORK_DIR, 'detect_signs.py')
with open(script_path, 'w', encoding='utf-8') as f:
    f.write(detect_script.strip())

print(f'Script PC sauvegarde : {script_path}')
print(f'\nPour l\'utiliser sur ton PC :')
print(f'  1. Telecharge depuis Drive :')
print(f'       - WORK/models/sign_language_final.h5')
print(f'       - WORK/model_config.json')
print(f'       - WORK/detect_signs.py')
print(f'  2. Dans un terminal :')
print(f'       pip install tensorflow mediapipe opencv-python numpy')
print(f'  3. Lance :')
print(f'       python detect_signs.py')

## RECAPITULATIF FINAL

In [ ]:
print('=' * 60)
print('  PROJET WLASL - RECAPITULATIF FINAL')
print('=' * 60)
print(f'  Signes entraines : {NUM_CLASSES}')
print(f'  Precision test   : {test_acc*100:.2f}%')
print(f'  Top-5 test       : {test_top5*100:.2f}%')
print(f'  Loss test        : {test_loss:.4f}')
print('=' * 60)
print(f'\nFichiers generes dans Drive/ASL_PROJECT/WORK/ :')
for root, dirs, files in os.walk(WORK_DIR):
    level  = root.replace(WORK_DIR, '').count(os.sep)
    indent = '  ' * (level + 1)
    print(f'{'  ' * level}{os.path.basename(root)}/')
    for fname in files:
        fpath   = os.path.join(root, fname)
        size_mb = os.path.getsize(fpath) / 1024 / 1024
        print(f'{indent}{fname}  ({size_mb:.1f} MB)')
print()
print('Tout est pret. Bonne detection !')